# Chapter 10: Tokenization and Data at Scale

**Companion notebook** for *Predicting the Next Words* (Osterrieder, 2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josterri/predicting-the-next-words-notebooks/blob/main/ch10_tokenization_and_data_at_scale.ipynb)


## What is in this notebook, and what it needs before it runs

Three cells on how text becomes tokens, and on what it costs when it does so
badly.

1. **BPE from scratch**, merge by merge, on a four-word corpus. About thirty
   lines of plain Python, and the clearest place in the pack to watch the
   algorithm actually run.
2. **SentencePiece trained on a small corpus** written to a temporary file.
3. **Fertility across four languages** with GPT-2's tokenizer: how many tokens
   the same meaning costs in English, German, Chinese and Korean.

Cell 2 needs `sentencepiece`, which is not installed here, so the notebook is
committed with no stored output. Cells 1 and 3 would run: cell 1 needs nothing
at all, and cell 3 needs GPT-2's tokenizer, which this machine already has
cached from chapter 1. To run the whole file: `pip install sentencepiece`. After
that nothing takes more than a few seconds.

Cell 1 is the one to change, and the change is to print more. Add a line inside
the merge loop that prints the pair being merged and its count, rerun, and read
the merges in order. There is no linguistic knowledge in BPE at all: it merges
whatever is frequent. Watching it choose is the quickest route to understanding
both why it works on English and why cell 3's fertility numbers come out the way
they do.


> **This notebook was not executed when it was built, so no cell below has
> stored output.** The reason: cell 2 needs `sentencepiece`, which is not installed on the machine this pack is built on. Cells 1 and 3 would run: cell 3's GPT-2 tokenizer is already cached here.
>
> Nothing here is broken. It is code to read now and to run once you have what
> it needs, and the section above says what that is. Build it yourself with
> `python tools/build_notebook.py ch10` on a machine that has them.


### 10.2.1 The BPE Algorithm Step by Step

The following implementation demonstrates BPE from scratch in Python, using only the `collections.Counter` class and a regular expression for merge application:


In [ ]:
# Colab does not ship these. Running this cell is a no-op if they are already present.
%pip install -q transformers sentencepiece


In [ ]:
import re
from collections import Counter

def get_pairs(vocab):
    """Count adjacent symbol pairs across all words."""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

# Training corpus: each word as space-separated characters + end marker
corpus = {"l o w </w>": 5, "l o w e r </w>": 2, "n e w </w>": 6,
          "n e w e r </w>": 3, "w i d e r </w>": 2}

print("Initial vocabulary:", sorted({c for w in corpus for c in w.split()}))
for step in range(10):  # 10 BPE merges
    pairs = get_pairs(corpus)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    print(f"Merge {step+1}: {best[0]}+{best[1]} -> {''.join(best)}"
          f" (freq={pairs[best]})")
    # Apply merge to entire corpus
    merged = re.escape(' '.join(best))
    replacement = ''.join(best)
    corpus = {re.sub(merged, replacement, w): f
              for w, f in corpus.items()}
# Result: vocabulary grows from characters to meaningful subwords


### 10.3.1 SentencePiece: Language-Independent Preprocessing


In [ ]:
import sentencepiece as spm
import tempfile, os

# Write a small training corpus, one sentence per line so SentencePiece
# can read each as a separate training example.
sentences = [
    "The cat sat on the mat.",
    "Language models predict the next word.",
    "Tokenization splits text into subwords.",
] * 100
corpus_file = os.path.join(tempfile.gettempdir(), "corpus.txt")
with open(corpus_file, "w", encoding="utf-8") as f:
    f.write("\n".join(sentences))

# Train BPE and Unigram tokenizers via SentencePiece. We request a
# small vocab_size (32) that is feasible for this tiny corpus; in
# practice values of 8,000-32,000 are typical.
for model_type in ["bpe", "unigram"]:
    prefix = os.path.join(tempfile.gettempdir(), f"sp_{model_type}")
    spm.SentencePieceTrainer.train(
        input=corpus_file, model_prefix=prefix,
        vocab_size=32, model_type=model_type)
    sp = spm.SentencePieceProcessor(model_file=f"{prefix}.model")
    text = "Tokenization determines what the model predicts"
    pieces = sp.encode(text, out_type=str)
    print(f"{model_type:>8s}: {pieces}")
# BPE and Unigram produce different segmentations of the same text


### 10.4.2 Multilingual Tokenization and the Fertility Problem


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Compare fertility across languages (same semantic content)
# For CJK scripts we count characters as semantic units (no whitespace
# word boundary); for space-delimited languages we count whitespace words.
texts = {
    "English":  ("Artificial intelligence is transforming the world.", "words"),
    "German":   ("Kuenstliche Intelligenz veraendert die Welt.", "words"),
    "Chinese":  ("\u4eba\u5de5\u667a\u80fd\u6b63\u5728\u6539\u53d8\u4e16\u754c", "chars"),
    "Korean":   ("\uc778\uacf5\uc9c0\ub2a5\uc774 \uc138\uacc4\ub97c \ubcc0\ud654\uc2dc\ud0a4\uace0 \uc788\ub2e4", "words"),
}
for lang, (text, unit) in texts.items():
    tokens = tokenizer.encode(text)
    units = len(text) if unit == "chars" else len(text.split())
    fertility = len(tokens) / units
    print(f"{lang:>10s}: {len(tokens):2d} tokens / {units} {unit} "
          f"= fertility {fertility:.2f}")
# English fertility is roughly 1.0-1.3; CJK and Korean scripts show
# the tokenization tax (often 3-10x higher) because GPT-2's merges
# were learned almost entirely on English byte pairs.


---

## Summary

This notebook demonstrated the key code examples from Chapter 10: Tokenization and Data at Scale. For the full mathematical exposition and discussion, refer to the textbook chapter.
